# Segments closure (reconstruction)

JAXTPC's standard event reconstruction — based on
[`closure/segments/run.py`](../../closure/segments/run.py). An event is
reconstructed as **N point charges** `[x,y,z,dE]`, optimized through the
differentiable simulator against the observed wire signals with the **Sobolev
loss**, **Adam** (per-parameter learning rates), and **MCMC relocation** of dead
(near-zero-energy) segments — clone an alive donor, split its energy
charge-conservingly, zero both Adam moments (3DGS-MCMC).

The optimizer is **oversampled** (`N_SEG` ≫ `N_TRUTH`, each carrying `e_scale =
N_TRUTH/N_SEG` of the energy): redundant segments die and get relocated — that's
what relocation is for. Uses the *real* `relocate_segments`. Self-contained
(synthetic truth); for real edepsim events use the script directly. Builds on
[`../gradients/optimization.ipynb`](../gradients/optimization.ipynb).


In [ ]:
# Resolve the repo root so imports and config/ paths work from any folder.
import os, sys
_d = os.path.abspath(os.getcwd())
while _d != os.path.dirname(_d) and not os.path.isdir(os.path.join(_d, 'config')):
    _d = os.path.dirname(_d)
sys.path.insert(0, _d); os.chdir(_d)


In [ ]:
import numpy as np, jax, jax.numpy as jnp, optax, matplotlib.pyplot as plt
from tools.simulation import DetectorSimulator
from tools.geometry import generate_detector
from tools.loader import build_deposit_data
from tools.losses import sobolev_loss_geomean_log1p, make_sobolev_weight
from closure.segments.run import relocate_segments  # the real MCMC relocation
detector = generate_detector('config/cubic_wireplane_config.yaml')
N_TRUTH = 400; N_SEG = 1200; DX_MM = 4.0      # oversample 3x -> relocation matters
PLANE_NAMES = ['U0','V0','Y0','U1','V1','Y1']


## 1. Truth event → target signals


In [ ]:
rng = np.random.RandomState(0)
truth_pos = np.stack([np.linspace(-180, -30, N_TRUTH), np.linspace(-60, 60, N_TRUTH),
                      np.linspace(-40, 70, N_TRUTH)], 1).astype(np.float32) * 10
truth_de = np.full(N_TRUTH, 2.2 * DX_MM/10, np.float32)
sim_truth = DetectorSimulator(detector, use_bucketed=False, total_pad=10000,
    response_chunk_size=10000, include_track_hits=False, recombination_model='modified_box')
cfg = sim_truth.config
deposits = build_deposit_data(truth_pos, truth_de, np.full(N_TRUTH, DX_MM/10, np.float32),
                              cfg, track_ids=np.zeros(N_TRUTH, np.int32))
resp, _, _ = sim_truth.process_event(deposits)
truth = [jnp.zeros((1,1))]*6; weights = [jnp.zeros((1,1))]*6; active = []
for (v,p), s in resp.items():
    s = jnp.asarray(s); idx = v*3+p; truth[idx] = s
    if jnp.any(s != 0):
        active.append(idx); H,W = s.shape; weights[idx] = make_sobolev_weight(H, W, s=1.0)
active = sorted(active); truth = tuple(truth); weights = tuple(weights)
print('active planes:', [PLANE_NAMES[i] for i in active], '| truth total dE = %.1f MeV' % truth_de.sum())


## 2. Differentiable forward + Sobolev loss


In [ ]:
sim_opt = DetectorSimulator(detector, differentiable=True, n_segments=N_SEG, total_pad=10000, recombination_model='modified_box')
sp = sim_opt.default_sim_params
def forward(params):
    return sim_opt.forward_segments(sp, params[:, :3], params[:, 3], dx=DX_MM)
def loss_fn(params):
    return sobolev_loss_geomean_log1p(forward(params), truth, weights, planes=tuple(active))
grad_fn = jax.jit(jax.value_and_grad(loss_fn))


## 3. Initialize (oversampled) + recombination constants
Subsample the truth point charges up to `N_SEG` with replacement and scale each by
`e_scale = N_TRUTH/N_SEG`. The charge-conserving split needs `(death_thresh, alpha,
B, dx_cm)`, derived from the recombination params as the closure script does.


In [ ]:
e_scale = N_TRUTH / N_SEG
idx = rng.choice(N_TRUTH, N_SEG, replace=True)
init = np.concatenate([truth_pos[idx] + rng.normal(0, 50, (N_SEG, 3)).astype(np.float32),
                       (truth_de[idx] * e_scale * rng.uniform(0.7, 1.3, N_SEG)).reshape(-1, 1)], 1)
params = jnp.asarray(init)
rp = sp.recomb_params
dens = float(rp.density); alpha_r = float(rp.alpha)
beta_r = float(getattr(rp, 'beta_90', getattr(rp, 'beta', 0.212)))
field_kVcm = float(getattr(rp, 'field_strength_Vcm', 500.0)) / 1000.0
B_eff = beta_r / dens / field_kVcm
DEATH_THRESH, MIN_E = 0.012, 0.001
recomb_constants = (DEATH_THRESH, alpha_r, B_eff, DX_MM/10.0)
print(f'e_scale={e_scale:.2f}, recomb split: alpha={alpha_r:.3f}, B={B_eff:.4f}, death={DEATH_THRESH} MeV')


## 4. Optimize with MCMC relocation
Adam (energy LR ×`lr_e_mult`, floored at `min_energy`) every step; after a warmup,
relocate dead segments every `RELOC_EVERY` steps.


In [ ]:
LR_E_MULT = 0.01; WARMUP = 30; RELOC_EVERY = 15; STEPS = 180; MAX_RELOC = N_SEG
optimizer = optax.adam(optax.exponential_decay(1.0, 1, 0.999), b1=0.9, b2=0.999)
opt_state = optimizer.init(params); key = jax.random.PRNGKey(0)
losses, dead_counts, relocs = [], [], []
for i in range(STEPS):
    L, g = grad_fn(params)
    upd, opt_state = optimizer.update(g, opt_state, params)
    upd = upd.at[:, 3].multiply(LR_E_MULT)
    params = optax.apply_updates(params, upd)
    params = params.at[:, 3].set(jnp.maximum(params[:, 3], MIN_E))
    n_reloc = 0
    if i >= WARMUP and i % RELOC_EVERY == 0:
        params, opt_state, key, n_reloc = relocate_segments(params, opt_state, key, recomb_constants, MAX_RELOC)
        n_reloc = int(n_reloc)
    losses.append(float(L)); relocs.append(n_reloc)
    dead_counts.append(int(jnp.sum(params[:, 3] <= DEATH_THRESH)))
    if i % 20 == 0 or i == STEPS - 1:
        print(f'  step {i:3d}   loss {float(L):.4f}   dead {dead_counts[-1]:4d}   relocated {n_reloc:4d}')
print(f'total relocations: {sum(relocs)}')


## 5. Results


In [ ]:
recon = forward(params); yp = 2
T = np.abs(np.asarray(truth[yp])); R = np.abs(np.asarray(recon[yp]))
vmax = np.percentile(T[T > 0], 99) if np.any(T > 0) else 1.0
fig, ax = plt.subplots(1, 4, figsize=(20, 4))
ax[0].plot(losses); ax[0].set(title='Sobolev loss', xlabel='step'); ax[0].grid(alpha=0.3)
ax[1].plot(dead_counts); ax[1].set(title='dead segments', xlabel='step'); ax[1].grid(alpha=0.3)
for a, img, ttl in [(ax[2], T, 'truth (Y)'), (ax[3], R, 'reconstruction (Y)')]:
    a.imshow(img.T, aspect='auto', origin='lower', cmap='inferno', vmin=0, vmax=vmax)
    a.set(title=ttl, xlabel='wire', ylabel='time')
fig.suptitle(f'Segments closure: loss {losses[0]:.3f} -> {losses[-1]:.3f}, {sum(relocs)} relocations')
plt.tight_layout(); plt.show()


## Next
- `mcs_closure.ipynb` — MCS muon (vertex/direction/energy + scattering angles)
- `muon_closure.ipynb` — muon track from initial guesses
- `closure/segments/run.py` — full method on real edepsim events
